# Setup

In [4]:
import sys, os
from pathlib import Path
import logging

def find_project_root(marker="backend", start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not find a '{marker}' folder above {current}")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "scripts"))

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)

from config.settings import settings
from medrag.embeddings.qdrant_client import get_qdrant_client
from medrag.retrieval.reranking import search_with_reranking

import openai

client = get_qdrant_client(settings.qdrant_url or "http://localhost:6333")
client.get_collections()

openai_client = openai.OpenAI(api_key=settings.openai_api_key)
 
GENERATION_MODEL = "gpt-4.1-nano"

print("Setup complete")

Project root: C:\Users\DELL\Desktop\medrag
Setup complete


# Plain grounded generation, test on a well-covered query

In [5]:
models = openai_client.models.list()
for m in sorted(models.data, key=lambda x: x.id):
    print(m.id)

chatgpt-image-latest
gpt-3.5-turbo
gpt-3.5-turbo-0125
gpt-3.5-turbo-1106
gpt-4
gpt-4-0613
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4.1-nano
gpt-4o-mini-transcribe
gpt-4o-mini-transcribe-2025-03-20
gpt-4o-mini-transcribe-2025-12-15
gpt-4o-mini-tts
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-codex
gpt-5-search-api
gpt-5-search-api-2025-10-14
gpt-5.1-chat-latest
gpt-5.1-codex
gpt-5.2
gpt-5.2-chat-latest
gpt-5.2-pro
gpt-5.2-pro-2025-12-11
gpt-audio-mini-2025-12-15
gpt-image-1
gpt-realtime-mini
gpt-realtime-mini-2025-12-15
o3-2025-04-16
text-embedding-3-large
text-embedding-3-small
text-embedding-ada-002
tts-1
tts-1-1106


In [6]:
BASE_SYSTEM_PROMPT = """You are a medical information assistant. Answer the user's question using ONLY the information in the provided context below. Do not use any outside knowledge.

If the context does not contain enough information to answer the question, say so plainly - do not guess or fill gaps with your own knowledge.

Context:
{context}
"""

def format_context(results: list) -> str:
    """Format reranked search results into a numbered context block."""
    blocks = []
    for i, r in enumerate(results, 1):
        blocks.append(f"[{i}] (source: {r['payload']['source']})\n{r['payload']['raw_text']}")
    return "\n\n".join(blocks)


def generate_answer(query: str, model: str = GENERATION_MODEL) -> str:
    results = search_with_reranking(client, query, candidate_pool_size=20, top_n=5)
    context = format_context(results)

    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": BASE_SYSTEM_PROMPT.format(context=context)},
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content


answer = generate_answer("side effects of ACE inhibitors")
print(answer)

Batches: 100%|██████████| 1/1 [00:04<00:00,  4.48s/it]


The major adverse effects of ACE inhibitors include dry cough, renal dysfunction in patients with impaired renal function, angioedema, hypotension, hyperkalemia, hypersensitivity reactions, and rare cases of cholestatic jaundice and hepatic failure.


In [7]:
answer = generate_answer("how does the body regulate blood sugar")
print(answer)

Batches: 100%|██████████| 1/1 [00:07<00:00,  7.87s/it]


The context does not provide specific details about how the body regulates blood sugar.


# Add citation instructions

In [8]:
CITED_SYSTEM_PROMPT = """You are a medical information assistant. Answer the user's question using ONLY the information in the provided context below. Do not use any outside knowledge.

If the context does not contain enough information to answer the question, say so plainly - do not guess or fill gaps with your own knowledge.

After each claim or statement in your answer, add a bracketed citation referencing which numbered context block it came from, e.g. "ACE inhibitors can cause dry cough [1]." If a claim is supported by multiple sources, cite all of them, e.g. [1][3].

Context:
{context}
"""

def generate_answer_cited(query: str, model: str = GENERATION_MODEL) -> str:
    results = search_with_reranking(client, query, candidate_pool_size=20, top_n=5)
    context = format_context(results)

    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": CITED_SYSTEM_PROMPT.format(context=context)},
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content


answer = generate_answer_cited("side effects of ACE inhibitors")
print(answer)

Batches: 100%|██████████| 1/1 [00:03<00:00,  3.97s/it]


The major adverse effects of ACE inhibitors include dry cough and renal dysfunction in patients with impaired renal function [1]. Additionally, ACE inhibitors have been associated with angioedema, hypovolemia, hypotension, hyperkalemia, and, more rarely, cholestatic jaundice, hepatic failure, neutropenia, and agranulocytosis [3].


# Add multilingual auto-detection

In [9]:
MULTILINGUAL_SYSTEM_PROMPT = """You are a medical information assistant. Answer the user's question using ONLY the information in the provided context below. Do not use any outside knowledge.

If the context does not contain enough information to answer the question, say so plainly - do not guess or fill gaps with your own knowledge.

The context below is in English. Detect the language of the user's question and respond in THAT SAME language, translating the relevant information from the English context as needed. If the question is in English, respond in English.

After each claim or statement in your answer, add a bracketed citation referencing which numbered context block it came from, e.g. "ACE inhibitors can cause dry cough [1]." If a claim is supported by multiple sources, cite all of them, e.g. [1][3].

Context:
{context}
"""

def generate_answer_full(query: str, model: str = GENERATION_MODEL) -> str:
    results = search_with_reranking(client, query, candidate_pool_size=20, top_n=5)
    context = format_context(results)

    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": MULTILINGUAL_SYSTEM_PROMPT.format(context=context)},
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content


# Test in Urdu (one of your 6 target languages)
answer = generate_answer_full("ACE inhibitors ke side effects kya hain?")
print(answer)

Batches: 100%|██████████| 1/1 [00:11<00:00, 11.05s/it]


ACE inhibitors ke major adverse effects dry cough aur renal dysfunction hain, khas kar un patients mein jin ki renal function impaired ho [1].


# Integrate Neo4j facts as a distinct, high-confidence context block

In [10]:
from neo4j import GraphDatabase

neo4j_driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password),
)
neo4j_driver.verify_connectivity()

def get_graph_facts_for_drug(drug_name: str) -> list:
    """Query Neo4j for all known relationships for a drug, matched by
    normalized (lowercase) name - same normalization used during Phase
    13 ingestion."""
    normalized = drug_name.strip().lower()
    with neo4j_driver.session() as session:
        result = session.run("""
            MATCH (d:Drug {normalized_name: $name})-[rel]->(dis:Disease)
            RETURN d.name AS drug, type(rel) AS relationship, dis.name AS disease
        """, name=normalized)
        return [dict(record) for record in result]


def format_graph_facts(facts: list) -> str:
    if not facts:
        return ""
    lines = [f"- {f['drug']} {f['relationship']} {f['disease']}" for f in facts]
    return "Verified structured facts from the medical knowledge graph:\n" + "\n".join(lines)


# Test: query for a known drug
facts = get_graph_facts_for_drug("Metformin Hydrochloride")
print(format_graph_facts(facts))

Verified structured facts from the medical knowledge graph:
- Metformin Hydrochloride CAUSES asthenia
- Metformin Hydrochloride CAUSES diarrhea
- Metformin Hydrochloride CAUSES hypoxemia
- Metformin Hydrochloride CAUSES type 2 diabetes mellitus
- Metformin Hydrochloride CAUSES acute congestive heart failure
- Metformin Hydrochloride CAUSES flushing, palpitation
- Metformin Hydrochloride CAUSES dyspnea
- Metformin Hydrochloride CAUSES cardiac impairment
- Metformin Hydrochloride CAUSES hepatic impairment
- Metformin Hydrochloride CAUSES nail disorder
- Metformin Hydrochloride CAUSES alcoholism
- Metformin Hydrochloride CAUSES hypoperfusion
- Metformin Hydrochloride CAUSES myalgia
- Metformin Hydrochloride CAUSES headache
- Metformin Hydrochloride CAUSES rash
- Metformin Hydrochloride CAUSES metformin-treated
- Metformin Hydrochloride CAUSES taste disorder
- Metformin Hydrochloride CAUSES shock
- Metformin Hydrochloride CAUSES prerenal azotemia
- Metformin Hydrochloride CAUSES macrovascu

# lean and cap graph facts before use in generation

In [11]:
def get_graph_facts_for_drug_curated(drug_name: str, max_causes: int = 8) -> list:
    """Same query as Cell 7, but curated before use in generation:
    - drops any CAUSES fact whose disease also appears under TREATS for
      the same drug (a direct contradiction - confirmed on real data,
      e.g. metformin CAUSES/TREATS type 2 diabetes mellitus - resolved
      in favor of TREATS, since indications_and_usage is the more
      authoritative, lower-noise section)
    - caps CAUSES to max_causes, since a drug can have dozens of listed
      adverse effects and including all of them would both bloat the
      prompt and bury the higher-value TREATS/CONTRAINDICATED_IN facts
    """
    facts = get_graph_facts_for_drug(drug_name)

    treats_diseases = {f["disease"].lower() for f in facts if f["relationship"] == "TREATS"}
    causes = [f for f in facts if f["relationship"] == "CAUSES" and f["disease"].lower() not in treats_diseases]
    treats = [f for f in facts if f["relationship"] == "TREATS"]
    contraindicated = [f for f in facts if f["relationship"] == "CONTRAINDICATED_IN"]

    return treats + contraindicated + causes[:max_causes]


facts_curated = get_graph_facts_for_drug_curated("Metformin Hydrochloride")
print(format_graph_facts(facts_curated))

Verified structured facts from the medical knowledge graph:
- Metformin Hydrochloride TREATS type 2 diabetes mellitus
- Metformin Hydrochloride CONTRAINDICATED_IN coma
- Metformin Hydrochloride CONTRAINDICATED_IN diabetic ketoacidosis
- Metformin Hydrochloride CONTRAINDICATED_IN metabolic acidosis
- Metformin Hydrochloride CONTRAINDICATED_IN Hypersensitivity
- Metformin Hydrochloride CONTRAINDICATED_IN renal impairment
- Metformin Hydrochloride CAUSES asthenia
- Metformin Hydrochloride CAUSES diarrhea
- Metformin Hydrochloride CAUSES hypoxemia
- Metformin Hydrochloride CAUSES acute congestive heart failure
- Metformin Hydrochloride CAUSES flushing, palpitation
- Metformin Hydrochloride CAUSES dyspnea
- Metformin Hydrochloride CAUSES cardiac impairment
- Metformin Hydrochloride CAUSES hepatic impairment


# Full pipeline: retrieval + graph facts + citation + multilingual

In [12]:
def get_all_known_drug_names() -> list:
    with neo4j_driver.session() as session:
        result = session.run("MATCH (d:Drug) RETURN d.name AS name")
        return [record["name"] for record in result]

KNOWN_DRUG_NAMES = get_all_known_drug_names()
print(f"Loaded {len(KNOWN_DRUG_NAMES)} known drug names")


def find_mentioned_drug(query: str) -> str:
    """Simple substring match against known drug names - not NER, since
    we already have the exact canonical name list from the graph
    itself, more reliable than re-extracting from the query text."""
    query_lower = query.lower()
    for drug_name in KNOWN_DRUG_NAMES:
        if drug_name.lower() in query_lower:
            return drug_name
    return None


FULL_SYSTEM_PROMPT = """You are a medical information assistant. Answer the user's question using ONLY the information in the provided context below. Do not use any outside knowledge.

If the context does not contain enough information to answer the question, say so plainly - do not guess or fill gaps with your own knowledge.

The context below is in English. Detect the language of the user's question and respond in THAT SAME language, translating the relevant information from the English context as needed. If the question is in English, respond in English.

After each claim or statement in your answer, add a bracketed citation referencing which numbered context block it came from, e.g. "ACE inhibitors can cause dry cough [1]." If a claim is supported by multiple sources, cite all of them, e.g. [1][3].

{graph_section}

Context:
{context}
"""

def generate_final_answer(query: str, model: str = GENERATION_MODEL) -> str:
    results = search_with_reranking(client, query, candidate_pool_size=20, top_n=5)
    context = format_context(results)

    mentioned_drug = find_mentioned_drug(query)
    graph_section = ""
    if mentioned_drug:
        facts = get_graph_facts_for_drug_curated(mentioned_drug)
        if facts:
            graph_section = format_graph_facts(facts) + "\n\nTreat the above as verified, high-confidence facts - if the retrieved context below conflicts with them, prioritize these facts."

    response = openai_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": FULL_SYSTEM_PROMPT.format(context=context, graph_section=graph_section)},
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content


answer = generate_final_answer("What does metformin treat and what are its contraindications?")
print(answer)

Loaded 466 known drug names


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.86s/it]


Metformin is an antihyperglycemic agent used to improve glucose tolerance in patients with type 2 diabetes mellitus. It lowers plasma glucose levels by decreasing hepatic glucose production, decreasing intestinal absorption of glucose, and improving insulin sensitivity [2][3].

The contraindications for metformin include severe renal impairment (eGFR below 30 mL/min/1.73 m^2), hypersensitivity to metformin, and acute or chronic metabolic acidosis, including diabetic ketoacidosis [1][4][5].
